# Day 4: Architecture and MongoDB
**Charts with Matplotlib, client server and API architecture, NoSQL, BSON and MongoDB CRUD**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup
Colab uses **mongomock**, an in-memory practice MongoDB. VS Code uses a real MongoDB (local or Atlas). The CRUD code is identical.

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q mongomock
import mongomock
client = mongomock.MongoClient()

# ===== Option B: VS Code with real MongoDB (uncomment to use) =====
# %pip install pymongo
# import os
# from pymongo import MongoClient
# uri = os.environ.get("MONGO_URI", "mongodb://localhost:27017")   # or your Atlas URI
# client = MongoClient(uri)

db = client["shop"]
products = db["products"]
products.delete_many({})       # start fresh each run
print("Connected. Collections:", db.list_collection_names())

## 2. Matplotlib basics

In [ ]:
import matplotlib.pyplot as plt

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
sales_2025 = [120, 150, 90, 180, 210, 190]
sales_2026 = [140, 160, 130, 200, 260, 240]

plt.figure(figsize=(7, 4))
plt.plot(months, sales_2025, marker="o", label="2025")
plt.plot(months, sales_2026, marker="o", label="2026")
plt.title("Monthly sales")
plt.xlabel("Month")
plt.ylabel("Units")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
cats = ["Food", "Rent", "Travel", "Fun"]
amounts = [6000, 15000, 2500, 3000]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
bars = ax1.bar(cats, amounts, color="#0FA3A3")
ax1.bar_label(bars)
ax1.set_title("Expenses by category")
ax2.pie(amounts, labels=cats, autopct="%1.0f%%")
ax2.set_title("Share of spending")
fig.tight_layout()
plt.savefig("expenses.png")
plt.show()

### TODO 1: Plot a histogram of these marks with 5 bins: `[45, 67, 72, 88, 91, 56, 73, 69, 80, 95, 61]`

In [ ]:
# TODO: write your code here

## 3. Create: insert documents

In [ ]:
result = products.insert_one({"name": "Pen", "category": "Stationery", "price": 20, "stock": 100})
print("New id:", result.inserted_id)

products.insert_many([
    {"name": "Notebook", "category": "Stationery", "price": 80, "stock": 40},
    {"name": "Mouse", "category": "Electronics", "price": 500, "stock": 15},
    {"name": "Keyboard", "category": "Electronics", "price": 1500, "stock": 4},
    {"name": "Monitor", "category": "Electronics", "price": 9000, "stock": 3},
    {"name": "Mug", "category": "Home", "price": 250, "stock": 0, "tags": ["gift", "kitchen"]},
    {"name": "Lamp", "category": "Home", "price": 1200, "stock": 7},
])
print("Total documents:", products.count_documents({}))

## 4. Read: find with filters and projections

In [ ]:
print(products.find_one({"name": "Mouse"}))

print("\nUnder 1000:")
for p in products.find({"price": {"$lt": 1000}}, {"_id": 0, "name": 1, "price": 1}):
    print(" ", p)

print("\nElectronics or Home with stock < 5:")
query = {"category": {"$in": ["Electronics", "Home"]}, "stock": {"$lt": 5}}
for p in products.find(query):
    print(" ", p["name"], p["stock"])

print("\nTop 3 most expensive:")
for p in products.find().sort("price", -1).limit(3):
    print(" ", p["name"], p["price"])

## 5. Update

In [ ]:
products.update_one({"name": "Pen"}, {"$set": {"price": 25}})
for p in products.find({"category": "Electronics"}):      # +10% price
    products.update_one({"_id": p["_id"]}, {"$set": {"price": round(p["price"] * 1.1)}})
products.update_many({"stock": 0}, {"$set": {"active": False}})
products.update_one({"name": "Mug"}, {"$push": {"tags": "ceramic"}})

for p in products.find({}, {"_id": 0, "name": 1, "price": 1, "active": 1, "tags": 1}):
    print(p)

## 6. Delete

In [ ]:
print("Will delete:", [p["name"] for p in products.find({"active": False})])   # check first!
res = products.delete_many({"active": False})
print("Deleted:", res.deleted_count, "| Remaining:", products.count_documents({}))

## 7. From database to chart

In [ ]:
value_by_cat = {}
for p in products.find():
    value_by_cat[p["category"]] = value_by_cat.get(p["category"], 0) + p["price"] * p["stock"]
print(value_by_cat)

plt.figure(figsize=(6, 4))
plt.bar(value_by_cat.keys(), value_by_cat.values(), color=["#0FA3A3", "#FF8A3D", "#0B5D63"])
plt.title("Stock value by category")
plt.ylabel("Value")
plt.show()

## 8. A layered design: repository class

In [ ]:
class ProductRepository:
    def __init__(self, collection):
        self.col = collection

    def add(self, product):
        return self.col.insert_one(product).inserted_id

    def by_category(self, category):
        return list(self.col.find({"category": category}, {"_id": 0}))

    def update_price(self, name, price):
        return self.col.update_one({"name": name}, {"$set": {"price": price}}).modified_count

repo = ProductRepository(products)
repo.add({"name": "Desk", "category": "Home", "price": 7000, "stock": 2})
print(repo.by_category("Home"))
print("Updated:", repo.update_price("Desk", 6500))

### TODO 2: Add a `delete_by_name` method to the repository and test it.

In [ ]:
# TODO: write your code here

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)